# Control policy optimization

In this example, a symbolic policy is evolved for the pendulum swingup task. Gymnax is used for simulation of the pendulum environment, showing that Kozax can easily be extended to external libraries.

In [ ]:
!pip install brax
!pip install gymnax
!pip install jaxtyping

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Specify the cores to use for XLA
import functools
import os
os.environ["XLA_FLAGS"] = '--xla_force_host_platform_device_count=10'
import jax
import jax.numpy as jnp
import jax.random as jr
import jaxlib

#import gymnax
import brax
from brax import envs
import matplotlib.pyplot as plt

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [ ]:
import jax
import jax.numpy as jnp
from brax import envs


class BraxGymnaxWrapper:

    def __init__(self, env_name, backend):
        self.env = envs.get_environment(env_name=env_name, backend=backend)
        self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, state, action, params=None):

      next_state = self.env.step(state, action)

      done = next_state.done

      # Freeze state after done
      obs = jnp.where(done, state.obs, next_state.obs)
      new_state = jax.tree.map(
          lambda new, old: jnp.where(done, old, new),
          next_state,
          state
      )

      reward = jnp.where(done, 0.0, next_state.reward)

      return obs, new_state, reward, done, {}

#Shouldn't states be represented the same in Gymnax and Brax -> env_state only used by respective environments to compute next step. Not explicitly used in Kozax
#EnvState(time=Array(0, dtype=int32, weak_type=True), theta=Array(1.4304566, dtype=float32), theta_dot=Array(0.5757351, dtype=float32), last_u=Array(0., dtype=float32, weak_type=True))) (Gymnax)
#State(pipeline_state, obs, reward, done, metrics, info)
#state.pipeline_state contains: q, qd, x, rot, xd, vel, contact, root_com, oinr, rot, i, mass, cd, vel, cdof, vel, cdofd, vel, mass_mx_inv, con_jac, con_aref, qf_smooth, qdd

#obs is the same for Gymnax and Brax

In [ ]:
from brax.envs.wrappers.training import EpisodeWrapper

In [ ]:
def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)
        (_, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn(policy),
            (obs, env_state, False),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

## Inverted pendulum

Action Space: [-1, 1], where action represents the numerical force applied to the cart

Observation Space: $ℝ^{4}$

0.   car_position
1.   vertical_angle_pole
2.   linear_velocity_cart
3.   angular_velocity_pole

Reward: +1 is awarded for each timestep that the pole is upright.

The episode terminates when episode duration reaches 1000 timestep or the absolute value of the vertical angle between the pole and the cart is greater than 0.2 radians.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("inverted_pendulum", 'generalized')
T = 1000

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  plot_list = []

  cart_y = 0.0
  pole_length = 0.5

  for i in range(0, T_term, 50):
    print(
            f"Timestep {i} : "
            f"Cart position: {all_obs[i, 0]} | "
            f"Cart velocity: {all_obs[i, 2]} | "
            f"Pole angle: {all_obs[i, 1]}"
        )
    x = all_obs[i, 0]
    theta = all_obs[i, 1]

    # Pole tip position
    pole_x = x + pole_length * jnp.sin(theta)
    pole_y = cart_y + pole_length * jnp.cos(theta)

    plot_list.append([i, x, pole_x, pole_y])


  print(
          f"Timestep {T_term} : "
          f"Cart position: {all_obs[T_term, 0]} | "
          f"Cart velocity: {all_obs[T_term, 2]} | "
          f"Pole angle: {all_obs[T_term, 1]}"
      )
  x = all_obs[T_term, 0]
  theta = all_obs[T_term, 1]

  # Pole tip position
  pole_x = x + pole_length * jnp.sin(theta)
  pole_y = cart_y + pole_length * jnp.cos(theta)

  plot_list.append([T_term, x, pole_x, pole_y])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(len(plot_list), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, pole_x, pole_y = plot_list[i]
    axs[i].scatter(x, cart_y, c="black")
    axs[i].plot([x, pole_x], [cart_y, pole_y])
    axs[i].scatter(pole_x, pole_y, c="red")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-2, 2)
    axs[i].set_ylim(-0.1, 1.0)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.plot(all_obs[:T_term, 0], label="car position")
  plt.plot(all_obs[:T_term, 1], label="pole angle")
  plt.xlabel("t")
  plt.legend()
  plt.show()

  # plt.plot(all_obs[:T_term, 1], all_obs[:T_term, 3])
  # plt.scatter(all_obs[0, 1], all_obs[0, 3], label="start point", zorder = 2)
  # plt.scatter(all_obs[T_term, 1], all_obs[T_term, 3], label="end point", zorder = 2)
  # plt.xlabel("Angle pole")
  # plt.ylabel("Angular velocity pole")
  # plt.title("Phase portrait t=1000")
  # plt.legend()
  # plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones)

### Multiple evaluation

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([0.1*obs[0]+obs[1]+obs[1]])) # y4*y0+y1+y1
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


## Inverted double pendulum

Action Space: [-1, 1], where action represents the numerical force applied to the cart

Observation Space: $ℝ^{4}$

0.   car_position
1.   sin_angle_cart_first_pole
2.   sin_angle_two_poles
3.   cos_angle_cart_first_pole
4.   cos_angle_two_poles
5.   car_velocity
6.   angular_velocity_cart_first_pole
7.   angular_velocity_two_poles

Reward: reward = alive_bonus - distance_penalty - velocity_penalty.

  - *alive_bonus*: +10 for each timestep that the pole is upright
  - *distance_penalty*: movement of tip of the second pendulum, calculated as $0.01 x_{pole2-tip}^2 + (y_{pole2-tip}-2)^2$.
  - *velocity_penalty*: Negative reward to penalize fast movement.
  $10^{-3} \omega_1 + 5 \times 10^{-3} \omega_2$,
  where $\omega_1, \omega_2$ are the angular velocities of the hinges.

The episode terminates when episode duration reaches 1000 timestep or the y_coordinate of the tip of the second pole $\leq 1$.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("inverted_double_pendulum", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  plot_list = []

  cart_y = 0.0
  pole_length = 0.5

  for i in range(0, T_term, 50):
    print(
            f"Timestep {i} : "
            f"Cart position: {all_obs[i, 0]} | "
            f"Cart velocity: {all_obs[i, 5]} | "
            f"Angle cart-first pole: {jnp.atan2(all_obs[i, 1], all_obs[i, 3])} |"
            f"Angle first pole-second pole: {jnp.atan2(all_obs[i, 2], all_obs[i, 4])} |"
        )
    x = all_obs[i, 0]
    sin_angle_1 = all_obs[i, 1]
    cos_angle_1 = all_obs[i, 3]
    sin_angle_2 = all_obs[i, 2]
    cos_angle_2 = all_obs[i, 4]

    # Pole1 tip position
    joint_x = x + pole_length * sin_angle_1
    joint_y = cart_y + pole_length * cos_angle_1

    # Pole1 tip position
    tip_x = joint_x + pole_length * sin_angle_2
    tip_y = joint_y + pole_length * cos_angle_2

    plot_list.append([i, x, joint_x, joint_y, tip_x, tip_y])


  print(
          f"Timestep {T_term} : "
          f"Cart position: {all_obs[T_term, 0]} | "
          f"Cart velocity: {all_obs[T_term, 5]} | "
          f"Angle cart-first pole: {jnp.atan2(all_obs[T_term, 1], all_obs[T_term, 3])} |"
          f"Angle first pole-second pole: {jnp.atan2(all_obs[T_term, 2], all_obs[T_term, 4])} |"
      )
  x = all_obs[i, 0]
  sin_angle_1 = all_obs[T_term, 1]
  cos_angle_1 = all_obs[T_term, 3]
  sin_angle_2 = all_obs[T_term, 2]
  cos_angle_2 = all_obs[T_term, 4]

  # Pole1 tip position
  joint_x = x + pole_length * sin_angle_1
  joint_y = cart_y + pole_length * cos_angle_1

  # Pole1 tip position
  tip_x = joint_x + pole_length * sin_angle_2
  tip_y = joint_y + pole_length * cos_angle_2

  plot_list.append([T_term, x, joint_x, joint_y, tip_x, tip_y])

  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(len(plot_list), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, joint_x, joint_y, tip_x, tip_y = plot_list[i]
    axs[i].scatter(x, cart_y, c="black")
    axs[i].plot([x, joint_x], [cart_y, joint_y])
    axs[i].scatter(joint_x, joint_y, c="black")
    axs[i].plot([joint_x, tip_x], [joint_y, tip_y])
    axs[i].scatter(tip_x, tip_y, c="red")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-2, 2)
    axs[i].set_ylim(-0.1, 1.0)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.plot(all_obs[:T_term, 0], label="cart position")
  plt.plot(all_obs[:T_term, 5], label="cart velocity")
  plt.xlabel("t")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones)

### Multiple evaluation

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([obs[1]-(obs[1]+obs[2]+obs[2])])) # [y1-(y1+y2+y2)]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[6085.7686 9352.39   7519.252  8480.27   6468.175  5534.562  7947.753
 9356.813  6077.8467 6985.178 ]


## Reacher

Action space: $[-1, 1]^2$, where the first action is the torque applied at the first hinge (connecting the link to the point of fixture), and the second action the torque applied at the second hinge (connecting the two links)

Observation space: $ℝ^{11}$

0.   cos(joint0)
1.   cos(joint1)
2.   sin(joint0)
3.   sin(joint1)
4.   target_x
5.   target_y
6.   ang_vel joint 0
7.   ang_vel joint 1
8.   diff x-value
9.   diff y-value
10.  (diff z-value)

The reward consists of two parts:

  - *reward_dist*: distance between *fingertip* of the reacher and the target,  with a more negative value assigned when *fingertip* is further away.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as the negative squared Euclidean norm of the action.


   The episode terminates when episode duration reaches a 1000 timesteps


### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("reacher", 'generalized')
T = 1000
#env = EpisodeWrapper(envs.get_environment("reacher"), episode_length=T, action_repeat=1)

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, new_state, reward, done, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, new_state, done.astype(bool)), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  all_list = []
  plot_list = []

  for i in range(1000):
    theta1 = jnp.arctan2(all_obs[i, 2], all_obs[i, 0])
    theta2 = jnp.arctan2(all_obs[i, 3], all_obs[i, 1])

    x1 = 0.1 * jnp.cos(theta1)
    y1 = 0.1 * jnp.sin(theta1)

    x2 = x1 + 0.1 * jnp.cos(theta1 + theta2)
    y2 = y1 + 0.1 * jnp.sin(theta1 + theta2)

    all_list.append([x2, y2])

    if i % 50 == 0:

      print(
              f"Timestep {i} : "
              f"x-coordinate tip: {x2} | "
              f"y-coordinate tip: {y2} | "
              f"Angular velocity joint 1: {all_obs[i, 6]} | "
              f"Angular velocity joint 2: {all_obs[i, 7]} | "
          )

      plot_list.append([i, x1, y1, x2, y2])

  nr_of_rows = 4
  nr_of_cols = 5

  fig, axs = plt.subplots(nrows=nr_of_rows, ncols=nr_of_cols, figsize=(12, 12))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    c = i % nr_of_cols
    r = int(i / nr_of_cols)
    t, x1, y1, x2, y2 = plot_list[i]
    axs[r, c].scatter(0,0, c = "black", label = "middle point")
    axs[r, c].scatter(x2, y2, c="blue", label = "arm tip")
    axs[r, c].plot([0, x1], [0, y1])
    axs[r, c].plot([x1, x2], [y1, y2])
    axs[r, c].scatter(all_obs[t, 4], all_obs[t, 5], c = "red", label = "target location")

    if i != 0:
      x2s_y2s = jnp.array(all_list[t-50:t])
      x2s = x2s_y2s[:, 0]
      y2s = x2s_y2s[:, 1]
      axs[r, c].plot(x2s, y2s, alpha = 0.4, label = "tip trajectory")

    axs[r, c].set_title("t="+ str(t))
    axs[r, c].set_xlim(-0.3, 0.3)
    axs[r, c].set_ylim(-0.3, 0.3)
    if r == 0 and c == nr_of_cols-1:
      axs[r, c].legend(loc='best', fontsize = 6)
    if i >= (nr_of_rows-1) * nr_of_cols:
      axs[r, c].set_xlabel("x")
    if c == 0:
      axs[r, c].set_ylabel("y")

  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_obs[:,8], alpha = 0.4, label='x-value diff')
  plt.plot(all_obs[:,9], alpha = 0.4, label='y-value diff')
  plt.plot(jnp.add(jnp.abs(all_obs[:, 8]), jnp.abs(all_obs[:, 9])), label="total value diff")
  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions)

### Multiple evaluation

In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([obs[10]-obs[7], obs[8]])) #[y10-y7, y8]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ -39.922165  -49.821873  -60.0979    -18.984558 -196.48444   -95.03198
  -85.89033   -64.415504 -143.82385   -35.902977]


## Swimmer

Action space: $[-1, 1]^2$, where the first action is the torque applied on the first rotor, and the second action the torque applied on the second rotor.

Observation space: $ℝ^{8}$

0.   angle_front_tip
1.   angle_second_rotor
2.   angle_third_rotor
3.   velocity_tip_x-axis
4.   velocity_tip_y-axis
5.   angular_velocity_front_tip
6.   angular_velocity_second_rotor
7.   angular_velocity_third_rotor

The reward consists of two parts:

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("swimmer", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, env_state, done.astype(bool)), (obs, reward, action, env_state.pipeline_state.x.pos)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_pos):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  plot_list = []

  for i in range(0, 1000, 50):
    print(
            f"Timestep {i} : "
            f"x-coordinate tip: {all_pos[i, -1, 0]} | "
            f"y-coordinate tip: {all_pos[i, -1, 1]} | "
        )
    xy = all_pos[i]
    plot_list.append([i, xy[:, 0], xy[:, 1]])

  nr_of_rows = 4
  nr_of_cols = 5

  fig, axs = plt.subplots(nrows=nr_of_rows, ncols=nr_of_cols, figsize=(12, 12))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, y = plot_list[i]
    c = i % nr_of_cols
    r = int(i / nr_of_cols)
    axs[r, c].plot(x, y)
    axs[r, c].scatter(x, y)
    x_till_t = all_pos[:t, -1, 0]
    y_till_t = all_pos[:t, -1, 1]
    axs[r, c].plot(x_till_t, y_till_t, label = "tip trajectory")
    axs[r, c].set_title("State at t="+ str(t))
    axs[r, c].set_xlim(-2,15)
    axs[r, c].set_ylim(-2,2)
    if i > (nr_of_rows-1) * nr_of_cols:
      axs[r, c].set_xlabel("x")
    if c == 0:
      axs[r, c].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  x = all_pos[:, -1, 0]
  y = all_pos[:, -1, 1]
  plt.plot(jnp.arange(1000), x, label = "x")
  plt.plot(jnp.arange(1000), y, label = "y")
  plt.xlabel("t")
  plt.ylabel("value")
  plt.legend()
  plt.show()

  plt.plot(x, y, label = "tip trajectory")
  plt.xlabel("x")
  plt.ylabel("y")
  plt.legend()
  plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, all_pos) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, all_pos)

### Multiple evaluation

In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([obs[6]-obs[2]+obs[5]+obs[4], obs[4]])) #[y6-y2+y5+y4, y4]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


[340.49014 354.60565 357.7974  356.29767 332.6057  343.8158  355.68326
 350.99344 350.90198 356.40247]


## Hopper

Action space: $[-1, 1]^3$, where the first action is the torque applied on the thigh rotor, and the second action the torque applied on the leg rotor, and the thrid action the torque applied to the foot rotor.

Observation space: $ℝ^{11}$

0.   z-coordinate of the top
1.   angle_top
2.   angle_thigh_joint
3.   angle_leg_joint
4.   angle_foot_joint
5.   velocity_x-coordinate_top
6.   velocity_z-coordinate_top
7.   angular_velocity_angle_top
8.   angular_velocity_thigh_hinge
9.   angular_velocity_leg_hinge
10.  angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*: +1 reward for every timestep that the hopper is alive.

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps, the height of the hopper becomes less than 0.7 metres, or the absolute value of the angle of the thigh joint is less than 0.2 radians

### Visualize

In [ ]:
env = BraxGymnaxWrapper("hopper", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            f"Height: {all_obs[i, 0]} | "
            f"x-coordinate thigh: {all_pos[i][2, 0]} | "
            f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate thigh: {all_pos[T_term][2, 0]} | "
          f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # if T_term < 1000:
  #   print("Height:", all_obs[T_term, 0], ", Angle (rad)", all_angles[T_term, 2])
  #   xz = all_pos[T_term][:, [0, 2]]
  #   plt.plot(xz[:, 0], xz[:, 1], "-o")
  #   plt.title("State at t="+ str(T_term))
  #   plt.xlim(-0.25, 0.25)
  #   plt.ylim(-0.2, 1.5)
  #   plt.show()

  #   plt.show()
  #   print("Terminated at timestep:", T_term)
  # else:
  #   print("Height:", all_obs[1000, 0], ", Angle (rad)", all_angles[1000, 2])
  #   xz = all_pos[1000][:, [0, 2]]
  #   plt.plot(xz[:, 0], xz[:, 1], "-o")
  #   plt.title("State at t="+ str(1000))
  #   plt.xlim(-0.25, 0.25)
  #   plt.ylim(-0.2, 1.5)
  #   plt.show()
  #   print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(max(6, len(plot_list)), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, xz = plot_list[i]
    axs[i].plot(xz[:, 0], xz[:, 1], "-o")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-0.25, 2)
    axs[i].set_ylim(-0.2, 1.5)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("z")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label=["action thigh", "action leg", "action foot"])
  plt.xlabel("t")
  plt.ylabel("value")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, all_dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles)

### Multiple evaluation

In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([0.1-obs[1]-obs[2], 0.1-obs[1], obs[9]])) #[0.1-y1-y2, 0.1-y1, y9]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1095.3315 1113.9839 1087.3208 1162.5234 1101.0757 1126.1235 1115.9884
 1135.301  1101.0874 1095.2422]


## Walker2d

Action space: $[-1, 1]^6$, where the first action is the torque applied on the thigh rotor, the second action the torque applied on the leg rotor, the third action the torque applied to the foot rotor, the fourth action is the torque applied on the left thigh rotor, the fifth action the torque applied on the left leg rotor, and the sixth action the torque applied to the left foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_top
1. angle_top
2. angle_thigh_joint
3. angle_leg_joint
4. angle_foot_joint
5. angle_left_thigh_joint
6. angle_left_leg_joing
7. angle_left_foot_joint
8. velocity_x-coordinate_top
9. velocity_z-coordinate_top
10. angular_velocity_top
11. angular_velocity_thigh_hinge
12. angular_velocity_leg_hinge
13. angular_velocity_foot_hinge
14. angular_velocity_thigh_hinge
15. angular_velocity_leg_hinge
16. angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*:
    +1 for each timepoint that the walker is alive
  - *reward_forward*:
    Reward of walking forward, measured as *(x-coordinate before action - x-coordinate after action) / dt*.
  - *reward_ctrl*:
    Negative reward for penalising the walker if it takes too large actions, measured as *-coefficient **x** sum(action<sup>2</sup>)*.

   The episode terminates when episode duration reaches a 1000 timesteps, when the height of the walker is not in the range [0.7, 2], and the absolute value of the angle is not in the range [-1, 1].

In [ ]:
def protected_divide(x1, x2):
    return jnp.where(x2 == 0, 1.0, x1 / x2)

def safe_power(x, y):
    return jnp.where(
        (x < 0) & (jnp.floor(y) != y),
        1.0,
        jnp.power(x, y)
    )

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("walker2d", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][2]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate: {all_angles[i][0]} | "
          f"z-coordinate: {all_angles[i][2]} | "
          #f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # for i in range(0, 1000, 10):
  #   xy = all_pos[i][:, :2]

  #   plt.plot(xy[:, 0], xy[:, 1])
  #   plt.scatter(xy[:, 0], xy[:, 1])
  #   plt.title("State at t="+ str(i))
  #   plt.gca().set_aspect("equal")
  #   plt.show()

  # xy = all_pos[:, -1, :]
  # x = xy[:, 0]
  # y = xy[:, 1]
  # plt.plot(jnp.arange(1000), x, label = "x")
  # plt.plot(jnp.arange(1000), y, label = "y")
  # plt.legend()
  # plt.show()
  # plt.plot(x, y, label = "tip trajectory")
  # plt.legend()
  # plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Multiple evaluation

In [ ]:
env = BraxGymnaxWrapper("walker2d", 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([1-(obs[1]+obs[0]), obs[7], jnp.cos(jnp.sin(0.1)), obs[12]-obs[10], obs[12]-obs[10], jnp.cos(obs[9])])) # [1-(y1+y0), y7, cos(sin(0.1)), y12-y10, y12-y10, cos(y9)]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, env)
print(reward_list_inv_pen)

[1006.7979   998.26044  186.08589 1002.9979  1002.74066  199.45091
  199.19106 1005.68115  188.41426  998.5209 ]


## Half cheetah

Action space: $[-1, 1]^6$, where the first action is the torque applied on the back thigh rotor, the second action the torque applied on the back shin rotor, the third action the torque applied to the back foot rotor, the fourth action is the torque applied on the front thigh rotor, the fifth action the torque applied on the front shin rotor, and the sixth action the torque applied to the front foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_mass
1. w-orientation_front_tip
2. y-orientation_front_tip
3. angle_back_thigh_rotor
4. angle_back_shin_rotor
5. angle_back_foot_rotor
6. velocity_tip_along_y-axis
7. angular_velocity_front_tip
8. angular_velocity_second_rotor
9. x-coordinate_front_tip
10. y-coordinate_front_tip
11. angle_front_tip
12. angle_second_rotor
13. angle_second_rotor
14. velocity_tip_along_x-axis
15. velocity_tip_along_y-axis
16. angular_velocity_front_tip
17. angular_velocity_second_rotor

The reward consists of two parts:

  - *reward_run*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("halfcheetah", 'generalized')
# env = EpisodeWrapper(envs.get_environment(env_name="halfcheetah"), episode_length=1000, action_repeat=1)
T = 1000

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )
        # env_state = env.step(
        #     env_state,
        #     action
        # )
        # reward = env_state.reward

        return (obs, env_state, done), (obs, reward, action, done) #, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][1]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate: {all_angles[i][0]} | "
          f"z-coordinate: {all_angles[i][1]} | "
          #f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # for i in range(0, 1000, 10):
  #   xy = all_pos[i][:, :2]

  #   plt.plot(xy[:, 0], xy[:, 1])
  #   plt.scatter(xy[:, 0], xy[:, 1])
  #   plt.title("State at t="+ str(i))
  #   plt.gca().set_aspect("equal")
  #   plt.show()

  # xy = all_pos[:, -1, :]
  # x = xy[:, 0]
  # y = xy[:, 1]
  # plt.plot(jnp.arange(1000), x, label = "x")
  # plt.plot(jnp.arange(1000), y, label = "y")
  # plt.legend()
  # plt.show()
  # plt.plot(x, y, label = "tip trajectory")
  # plt.legend()
  # plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Multiple evaluation

In [ ]:
env = BraxGymnaxWrapper("halfcheetah", 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([obs[12], obs[4], obs[12]-(obs[5]-obs[7]), obs[9]-obs[12], obs[11], obs[4]])) #[y12, y4, y12-(y5-y7), y9-y12, y11, y4]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, env)
print(reward_list_inv_pen)

[2996.433  2989.0247 2931.519  2966.6746 2851.7498 2970.5422 2968.7727
 3031.474  2988.5461 2880.8806]


## Ant

Action space: $[-1, 1]^8$, where the first action is the torque applied on the rotor between the torso and front left hip, the second action the torque applied on the rotor between the front left two links, the third action is the torque applied on the rotor between the torso and front right hip, the fourth action the torque applied on the rotor between the front right two links, the fifth action is the torque applied on the rotor between the torso and back left hip, the sixth action the torque applied on the rotor between the back left two links, the seventh action is the torque applied on the rotor between the torso and back right hip, the eighth action the torque applied on the rotor between the back right two links.

Observation space: $ℝ^{27}$

0. z-coordinate of torso
1. w-orientation_front_tip
2. x-orientation_front_tip
3. y-orientation_front_tip
4. z-orientation_front_tip
5. angle_torso_first_link_front_left
6. angle_two_links_front_left
7. angle_torso_first_link_front_right
8. angle_two_links_front_right
9. angle_torso_first_link_back_left
10. angle_two_links_back_left
11. angle_torso_first_link_back_right
12. angle_two_links_back_right
13. x-coordinate_velocity_torso
14. y-coordinate_velocity_torso
15. z-coordinate_velocity_torso
16. x-coordinate_angular_velocity_torso
17. y-coordinate_angular_velocity_torso
18. z-coordinate_angular_velocity_torso
19. angular_velocity_torso_front_left_link
20. angular_velocity_front_left_links
21. angular_velocity_torso_front_right_link
22. angular_velocity_front_right_links
23. angular_velocity_torso_back_left_link
24. angular_velocity_back_left_links
25. angular_velocity_torso_back_right_link
26. angular_velocity_back_right_links

The reward consists of four parts:

  - *reward_survive*: Every timestep that the ant is alive, it gets a reward of 1.

  - *reward_forward*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*

  - *contact_cost*: A negative reward for penalising too large external contact force. Calculated as calculated *0.5 * 0.001 * sum(clip(external contact force to [-1,1])<sup>2</sup>)*  


   The episode terminates when episode duration reaches a 1000 timesteps or when the y-orientation (index 3) in the state is **not** in the range `[0.2, 1.0]`.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("ant", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done) #, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_pos[i][1, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][2]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[i, 0]} | "
          f"x-coordinate: {all_pos[T_term][1, 0]} | "
          f"x-coordinate: {all_angles[T_term][0]} | "
          f"z-coordinate: {all_angles[T_term][2]} | "
          #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Multiple evaluation

In [ ]:
env = BraxGymnaxWrapper("ant", 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([jnp.cos(obs[12]), 0.1, obs[12], 1, obs[2], 0.1, jnp.cos(obs[12]), jnp.sin(obs[5])])) #[cos(y12), 0.1, y12, 1, y2, 0.1, cos(y12), sin(y5)]
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, env)
print(reward_list_inv_pen)

[ 480.9499    975.9896    158.4743    146.41995   931.418    1125.1726
  638.22107  1233.552     652.7863      6.751278]
